# text classification with knowledge distillation

exploring how knowledge distillation works for sentiment analysis. going to build a big teacher model and see if a tiny student can learn from it.

goals:
- load the imdb movie review dataset
- build teacher (big bidirectional lstm) 
- build student (small simple lstm)
- train both and compare them
- see if distillation actually helps

## a note on the stored outputs

The outputs of this notebook were cleared deliberately.
They were stale, they came from an earlier synthetic-data version of the notebook, and they contradicted both the current source and each other.

What was wrong with them, recorded here so the evidence is not lost along with the outputs:

- Stored output reported `vocabulary size: 62`, while the cell above it reported `vocabulary size: 10000` and the current source loads the 10,000-word IMDB set with `num_words=10000`.
- The two summary cells disagreed about model size. The size comparison cell reported a 2,766,146-parameter teacher and a 675,170-parameter student, the final summary cell reported 1,494,082 and 39,138 for the same two models.
- Teacher, distilled student and control student all reported exactly 100.00% accuracy, which is not a plausible IMDB score for any of them, let alone all three.
- The distillation training loop cell ended in a `KeyboardInterrupt` traceback, so that run never finished.
- Cell execution counts ran out of order (25, then 61 through 68, then 31, 32, 41, 44), so the stored state was never produced by one top-to-bottom pass.

TensorFlow is not installed in the environment this was cleared from and has no wheel for the Python version available there, so the notebook could not simply be re-run.
No source cell was changed. The distillation code is the same maths as the MNIST notebook and is believed sound.

This notebook needs a clean top-to-bottom re-run on a machine with TensorFlow before any number in it is quoted.


## imports and setup

In [ ]:
# using keras instead of pytorch (had issues with torch on this machine)
# uncomment if you need to install
#!pip install tensorflow

import numpy as np
import matplotlib.pyplot as plt
from tensorflow import keras
import tensorflow as tf
from collections import Counter
import time

np.random.seed(42)
tf.random.set_seed(42)

print(f"tensorflow version: {tf.__version__}")

## loading real imdb movie review dataset

using the famous imdb dataset - 50k movie reviews labeled as positive or negative. way better than fake data!

In [ ]:
# imdb: 25k train and 25k test reviews, binary sentiment, already tokenized to word indices
from tensorflow.keras.datasets import imdb

# keep the 10k most frequent words, rarer ones become the out of vocabulary id 2
max_features = 10000
(x_train_raw, y_train), (x_test_raw, y_test) = imdb.load_data(num_words=max_features)

print(f"training samples: {len(x_train_raw)}")
print(f"test samples: {len(x_test_raw)}")
print(f"classes: {len(set(y_train))} (0=negative, 1=positive)")

# both splits are balanced, so plain accuracy is a fair metric
from collections import Counter
train_dist = Counter(y_train)
test_dist = Counter(y_test)
print(f"\ntrain distribution: {dict(train_dist)}")
print(f"test distribution: {dict(test_dist)}")

In [ ]:
# get the word index mapping to convert back to text
word_index = imdb.get_word_index()
reverse_word_index = {value: key for key, value in word_index.items()}

def decode_review(encoded_review):
    """convert sequence of integers back to text"""
    # load_data shifts word indices by 3: 0 is padding, 1 marks the start, 2 is out of vocabulary
    decoded = ' '.join([reverse_word_index.get(i - 3, '?') for i in encoded_review])
    return decoded

print("first positive review:")
pos_idx = list(y_train).index(1)
print(f"encoded: {x_train_raw[pos_idx][:20]}...")
print(f"decoded: {decode_review(x_train_raw[pos_idx])[:200]}...")
print(f"label: {y_train[pos_idx]} (positive)")

print("\nfirst negative review:")
neg_idx = list(y_train).index(0)  
print(f"encoded: {x_train_raw[neg_idx][:20]}...")
print(f"decoded: {decode_review(x_train_raw[neg_idx])[:200]}...")
print(f"label: {y_train[neg_idx]} (negative)")

In [ ]:
# pad sequences to same length for batching
from tensorflow.keras.preprocessing.sequence import pad_sequences

train_lengths = [len(seq) for seq in x_train_raw]
test_lengths = [len(seq) for seq in x_test_raw]

print(f"sequence length stats:")
print(f"train - min: {min(train_lengths)}, max: {max(train_lengths)}, avg: {np.mean(train_lengths):.1f}")
print(f"test - min: {min(test_lengths)}, max: {max(test_lengths)}, avg: {np.mean(test_lengths):.1f}")

# choose a reasonable max length (covers most reviews)
max_len = 500  # covers about 92% of reviews
print(f"\nusing max_len: {max_len}")

x_train = pad_sequences(x_train_raw, maxlen=max_len, padding='post', truncating='post')
x_test = pad_sequences(x_test_raw, maxlen=max_len, padding='post', truncating='post')

print(f"final shapes:")
print(f"x_train: {x_train.shape}")
print(f"x_test: {x_test.shape}")
print(f"y_train: {y_train.shape}")
print(f"y_test: {y_test.shape}")

# embedding table size: kept word ids run from 0 to max_features - 1
vocab_size = max_features
print(f"vocabulary size: {vocab_size}")

## explore the real dataset

checking out the imdb reviews and their characteristics

In [ ]:
print("positive review examples:")
pos_indices = [i for i, label in enumerate(y_train) if label == 1][:3]
for i, idx in enumerate(pos_indices):
    review = decode_review(x_train_raw[idx])
    print(f"\n{i+1}. {review[:300]}...")

print("\n" + "="*50)
print("negative review examples:")
neg_indices = [i for i, label in enumerate(y_train) if label == 0][:3]
for i, idx in enumerate(neg_indices):
    review = decode_review(x_train_raw[idx])
    print(f"\n{i+1}. {review[:300]}...")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

axes[0,0].hist(train_lengths, bins=50, edgecolor='black', alpha=0.7, color='steelblue')
axes[0,0].axvline(max_len, color='red', linestyle='--', linewidth=2, label=f'max_len={max_len}')
axes[0,0].set_title('training sequence lengths', fontweight='bold')
axes[0,0].set_xlabel('sequence length')
axes[0,0].set_ylabel('frequency')
axes[0,0].legend()
axes[0,0].grid(True, alpha=0.3)

axes[0,1].hist(test_lengths, bins=50, edgecolor='black', alpha=0.7, color='orange')
axes[0,1].axvline(max_len, color='red', linestyle='--', linewidth=2, label=f'max_len={max_len}')
axes[0,1].set_title('test sequence lengths', fontweight='bold')
axes[0,1].set_xlabel('sequence length')
axes[0,1].set_ylabel('frequency')
axes[0,1].legend()
axes[0,1].grid(True, alpha=0.3)

train_labels, train_counts = zip(*train_dist.items())
test_labels, test_counts = zip(*test_dist.items())

axes[1,0].bar(['negative', 'positive'], train_counts, color=['red', 'green'], alpha=0.7, edgecolor='black')
axes[1,0].set_title('training class distribution', fontweight='bold')
axes[1,0].set_ylabel('count')
axes[1,0].grid(True, alpha=0.3, axis='y')

axes[1,1].bar(['negative', 'positive'], test_counts, color=['red', 'green'], alpha=0.7, edgecolor='black')
axes[1,1].set_title('test class distribution', fontweight='bold')
axes[1,1].set_ylabel('count')
axes[1,1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print(f"coverage with max_len={max_len}:")
print(f"train: {sum(1 for l in train_lengths if l <= max_len)/len(train_lengths)*100:.1f}%")
print(f"test: {sum(1 for l in test_lengths if l <= max_len)/len(test_lengths)*100:.1f}%")

In [ ]:
# word_index ranks words by corpus frequency, rank 1 is the most common word
word_freqs = [(word, idx) for word, idx in word_index.items()]
word_freqs.sort(key=lambda x: x[1])

top_words = word_freqs[:20]
words, indices = zip(*top_words)

plt.figure(figsize=(12, 6))
plt.barh(range(len(words)), [max_features - idx for idx in indices], color='steelblue')
plt.yticks(range(len(words)), words)
plt.xlabel('frequency rank (lower = more common)')
plt.title('top 20 most frequent words in imdb dataset', fontweight='bold')
plt.gca().invert_yaxis()
plt.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

print(f"total vocabulary size in imdb: {len(word_index)}")
print(f"using top {max_features} words")
print(f"coverage: {max_features/len(word_index)*100:.1f}% of vocabulary")

## data preprocessing complete

the imdb dataset comes pre-tokenized, so we just need to pad sequences for batching

In [ ]:
# imdb arrives tokenized, so padding was the only preprocessing step
print("data ready for training:")
print(f"x_train shape: {x_train.shape}")
print(f"x_test shape: {x_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")
print(f"vocab size: {vocab_size}")
print(f"max sequence length: {max_len}")

example_idx = 0
print(f"\nexample padded sequence (first 20 tokens):")
print(f"original length: {len(x_train_raw[example_idx])}")
print(f"padded sequence: {x_train[example_idx][:20]}")
print(f"decoded: {decode_review(x_train[example_idx][:20])}")

## build the teacher model

making a big bidirectional lstm with lots of parameters

In [ ]:
def create_teacher_model(vocab_size, max_len):
    model = keras.Sequential([
        keras.layers.Embedding(vocab_size, 128, input_length=max_len),
        keras.layers.Bidirectional(keras.layers.LSTM(256, return_sequences=True, dropout=0.3)),
        keras.layers.Bidirectional(keras.layers.LSTM(128, dropout=0.3)),
        keras.layers.Dense(128, activation='relu'),
        keras.layers.Dropout(0.5),
        keras.layers.Dense(64, activation='relu'),
        keras.layers.Dropout(0.3),
        keras.layers.Dense(2, activation='softmax')
    ])
    return model

# teacher: two stacked bidirectional lstms, each review is read in both directions
teacher = create_teacher_model(vocab_size, max_len)
teacher.summary()

## build the student model

way smaller - just one lstm layer and fewer params

In [ ]:
def create_student_model(vocab_size, max_len):
    model = keras.Sequential([
        keras.layers.Embedding(vocab_size, 64, input_length=max_len),
        keras.layers.LSTM(64, dropout=0.3),
        keras.layers.Dense(32, activation='relu'),
        keras.layers.Dropout(0.3),
        keras.layers.Dense(2, activation='softmax')
    ])
    return model

# student: one unidirectional lstm and a smaller embedding
student = create_student_model(vocab_size, max_len)
student.summary()

## compare model sizes

checking how much smaller the student is

In [ ]:
# build the models first so we can count params
teacher.build(input_shape=(None, max_len))
student.build(input_shape=(None, max_len))

teacher_params = teacher.count_params()
student_params = student.count_params()


print("model size comparison:")
print(f"teacher: {teacher_params:,} parameters")
print(f"student: {student_params:,} parameters")
print(f"compression ratio: {teacher_params/student_params:.1f}x")
print(f"size reduction: {(1-student_params/teacher_params)*100:.1f}%")

fig, ax = plt.subplots(figsize=(8, 5))
models = ['teacher\n(bidirectional)', 'student\n(simple)']
params = [teacher_params/1000, student_params/1000]  # in thousands
colors = ['#e74c3c', '#3498db']

bars = ax.bar(models, params, color=colors, alpha=0.7, edgecolor='black')
ax.set_ylabel('parameters (thousands)')
ax.set_title('model size comparison', fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

for bar, param in zip(bars, params):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{param:.1f}k',
            ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

## train the teacher

training the big model first so we can use it to teach the student

In [ ]:
print("training teacher model...")

In [ ]:
teacher.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# the teacher is trained first, on hard labels only, and is frozen during distillation
history_teacher = teacher.fit(x_train, y_train,validation_data=(x_test, y_test),epochs=5,batch_size=32,verbose=1)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(history_teacher.history['loss'], label='train loss', linewidth=2)
axes[0].plot(history_teacher.history['val_loss'], label='val loss', linewidth=2)
axes[0].set_title('teacher model loss', fontweight='bold')
axes[0].set_xlabel('epoch')
axes[0].set_ylabel('loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history_teacher.history['accuracy'], label='train acc', linewidth=2)
axes[1].plot(history_teacher.history['val_accuracy'], label='val acc', linewidth=2)
axes[1].set_title('teacher model accuracy', fontweight='bold')
axes[1].set_xlabel('epoch')
axes[1].set_ylabel('accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

teacher_acc = history_teacher.history['val_accuracy'][-1]
print(f"\nfinal teacher accuracy: {teacher_acc*100:.2f}%")

## train student without distillation (baseline)

first lets see how well the student does on its own

In [ ]:
student_baseline = create_student_model(vocab_size, max_len)

student_baseline.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history_baseline = student_baseline.fit(
    x_train, y_train,
    validation_data=(x_test, y_test),
    epochs=5,  # note: full 25k set for 5 epochs, the distilled student below sees 5k for 3
    batch_size=32,
    verbose=1
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(history_baseline.history['loss'], label='train loss', linewidth=2)
axes[0].plot(history_baseline.history['val_loss'], label='val loss', linewidth=2)
axes[0].set_title('student baseline loss', fontweight='bold')
axes[0].set_xlabel('epoch')
axes[0].set_ylabel('loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history_baseline.history['accuracy'], label='train acc', linewidth=2)
axes[1].plot(history_baseline.history['val_accuracy'], label='val acc', linewidth=2)
axes[1].set_title('student baseline accuracy', fontweight='bold')
axes[1].set_xlabel('epoch')
axes[1].set_ylabel('accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

baseline_acc = history_baseline.history['val_accuracy'][-1]
print(f"\nstudent baseline accuracy: {baseline_acc*100:.2f}%")

## implement knowledge distillation

now teaching the student using soft targets from the teacher

In [ ]:
def distillation_loss(y_true, y_pred_student, y_pred_teacher, temperature=4.0, alpha=0.7):
    """
    hinton style distillation loss: alpha * soft + (1 - alpha) * hard
    alpha is the weight on matching the teacher, 1 - alpha the weight on the true labels
    """
    # hard term: cross entropy with the true labels
    hard_loss = keras.losses.sparse_categorical_crossentropy(y_true, y_pred_student)
    
    # soft term: a temperature above 1 flattens the teacher's distribution so the small
    # probabilities it gives the wrong class carry signal, times t^2 to undo the 1 / t^2 gradient shrink
    # note: both models end in softmax, so these inputs are probabilities, not logits
    teacher_soft = tf.nn.softmax(y_pred_teacher / temperature)
    student_soft = tf.nn.softmax(y_pred_student / temperature)
    
    # manual kl divergence: kl(p||q) = sum(p * log(p/q))
    soft_loss = tf.reduce_sum(teacher_soft * tf.math.log(teacher_soft / (student_soft + 1e-8)), axis=1)
    soft_loss = tf.reduce_mean(soft_loss) * (temperature ** 2)
    
    total_loss = alpha * soft_loss + (1 - alpha) * tf.reduce_mean(hard_loss)
    return total_loss

print("distillation loss function ready")

## train student with distillation

using the teacher's knowledge to improve student learning

In [ ]:
student_distilled = create_student_model(vocab_size, max_len)

# use a subset of data for faster training (demo purposes)
subset_size = 5000
x_train_sub = x_train[:subset_size]
y_train_sub = y_train[:subset_size]

print(f"using subset of {subset_size} samples for faster training")

# t=4 gives moderately soft targets, alpha=0.7 puts most of the weight on the teacher
temperature = 4.0
alpha = 0.7
epochs = 3  # reduced further for demo
batch_size = 64  # larger batch for efficiency

optimizer = keras.optimizers.Adam()

train_losses = []
val_losses = []
val_accuracies = []

print(f"starting knowledge distillation training with imdb subset...")
print(f"temperature: {temperature}, alpha: {alpha}, epochs: {epochs}")
print(f"training samples: {len(x_train_sub)}, batch_size: {batch_size}")

In [ ]:
# custom loop because fit() has no hook for feeding teacher outputs into the loss

print("starting distillation training...")

for epoch in range(epochs):
    epoch_loss = 0
    num_batches = len(x_train_sub) // batch_size
    
    for i in range(num_batches):
        start_idx = i * batch_size
        end_idx = start_idx + batch_size
        
        batch_x = x_train_sub[start_idx:end_idx]
        batch_y = y_train_sub[start_idx:end_idx]
        
        with tf.GradientTape() as tape:
            # inference mode keeps the teacher's dropout off, it is frozen because only student variables get gradients
            teacher_pred = teacher(batch_x, training=False)
            student_pred = student_distilled(batch_x, training=True)
            
            loss = distillation_loss(batch_y, student_pred, teacher_pred, temperature, alpha)
        
        gradients = tape.gradient(loss, student_distilled.trainable_variables)
        optimizer.apply_gradients(zip(gradients, student_distilled.trainable_variables))
        
        epoch_loss += loss.numpy()
        
        if i % 20 == 0:
            print(f"  batch {i+1}/{num_batches} - loss: {loss.numpy():.4f}")
    
    # evaluated at t=1 on the test set, val_loss is hard label cross entropy only
    val_pred = student_distilled(x_test, training=False)
    val_loss = keras.losses.sparse_categorical_crossentropy(y_test, val_pred)
    val_loss = tf.reduce_mean(val_loss).numpy()
    
    val_acc = tf.reduce_mean(
        tf.cast(tf.equal(tf.argmax(val_pred, axis=1), y_test), tf.float32)
    ).numpy()
    
    train_losses.append(epoch_loss / num_batches)
    val_losses.append(val_loss)
    val_accuracies.append(val_acc)
    
    print(f"epoch {epoch+1}/{epochs} - loss: {epoch_loss/num_batches:.4f} - val_loss: {val_loss:.4f} - val_acc: {val_acc:.4f}")

print("distillation training completed!")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(train_losses, label='train loss', linewidth=2)
axes[0].plot(val_losses, label='val loss', linewidth=2)
axes[0].set_title('student distilled loss', fontweight='bold')
axes[0].set_xlabel('epoch')
axes[0].set_ylabel('loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(val_accuracies, label='val acc', linewidth=2, color='green')
axes[1].set_title('student distilled accuracy', fontweight='bold')
axes[1].set_xlabel('epoch')
axes[1].set_ylabel('accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

distilled_acc = val_accuracies[-1]
print(f"\nstudent distilled accuracy: {distilled_acc*100:.2f}%")

## compare all models


In [ ]:
models_comparison = {
    'teacher': teacher_acc,
    'student\n(baseline)': baseline_acc,
    'student\n(distilled)': distilled_acc}

fig, ax = plt.subplots(figsize=(10, 6))
model_names = list(models_comparison.keys())
accuracies = [models_comparison[m] * 100 for m in model_names]
colors = ['#e74c3c', '#95a5a6', '#27ae60']

bars = ax.bar(model_names, accuracies, color=colors, alpha=0.7, edgecolor='black', linewidth=2)
ax.set_ylabel('accuracy (%)', fontsize=12)
ax.set_title('model performance comparison', fontweight='bold', fontsize=14)
ax.set_ylim([0, 100])
ax.grid(True, alpha=0.3, axis='y')

for bar, acc in zip(bars, accuracies):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height + 1,
            f'{acc:.2f}%',
            ha='center', va='bottom', fontweight='bold', fontsize=11)

plt.tight_layout()
plt.show()

print("\nresults summary:")
print(f"teacher accuracy: {teacher_acc*100:.2f}%")
print(f"student baseline: {baseline_acc*100:.2f}%")
print(f"student distilled: {distilled_acc*100:.2f}%")
print(f"\nimprovement from distillation: {(distilled_acc - baseline_acc)*100:.2f} percentage points")

## test on some examples

trying the models on actual text to see how they predict

In [ ]:
# hand written reviews, a quick qualitative check outside the imdb test set
test_examples_raw = [
    "this movie was absolutely fantastic with amazing acting and plot",
    "worst film ever terrible acting boring story waste of time",
    "pretty good movie enjoyed most parts but ending was weak", 
    "not the best but had some decent moments worth watching",
    "horrible cinematography poor dialogue would not recommend",
    "brilliant masterpiece excellent direction superb performances"
]

print("converting test examples to sequences...")

In [ ]:
# note: load_data shifts word ids by 3 (index_from=3) and adds a start id 1, this conversion does neither,
# so these ids point at different words than the ones the models were trained on
def text_to_sequence(text, word_index, max_features):
    """convert text to sequence of integers"""
    words = text.lower().split()
    sequence = []
    for word in words:
        if word in word_index:
            idx = word_index[word]
            if idx < max_features:  # only use words in our vocab
                sequence.append(idx)
    return sequence

test_sequences = [text_to_sequence(text, word_index, max_features) for text in test_examples_raw]
test_padded = pad_sequences(test_sequences, maxlen=max_len, padding='post')

print("test sequences created:")
for i, (text, seq) in enumerate(zip(test_examples_raw, test_sequences)):
    print(f"{i+1}. '{text}' -> {len(seq)} tokens")

teacher_preds = teacher.predict(test_padded, verbose=0)
baseline_preds = student_baseline.predict(test_padded, verbose=0)
distilled_preds = student_distilled.predict(test_padded, verbose=0)

print("\npredictions on movie reviews:")
print("="*80)
for i, text in enumerate(test_examples_raw):
    print(f"\nreview: '{text}'")
    print(f"  teacher: {'positive' if np.argmax(teacher_preds[i]) == 1 else 'negative'} (conf: {np.max(teacher_preds[i]):.3f})")
    print(f"  baseline: {'positive' if np.argmax(baseline_preds[i]) == 1 else 'negative'} (conf: {np.max(baseline_preds[i]):.3f})")
    print(f"  distilled: {'positive' if np.argmax(distilled_preds[i]) == 1 else 'negative'} (conf: {np.max(distilled_preds[i]):.3f})")

## analyze prediction confidence

checking how confident each model is in its predictions

In [ ]:
# confidence is the probability of the predicted class
teacher_test_preds = teacher.predict(x_test, verbose=0)
baseline_test_preds = student_baseline.predict(x_test, verbose=0)
distilled_test_preds = student_distilled.predict(x_test, verbose=0)

teacher_conf = np.max(teacher_test_preds, axis=1)
baseline_conf = np.max(baseline_test_preds, axis=1)
distilled_conf = np.max(distilled_test_preds, axis=1)

In [ ]:


fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(teacher_conf, bins=20, color='#e74c3c', alpha=0.7, edgecolor='black')
axes[0].set_title('teacher confidence', fontweight='bold')
axes[0].set_xlabel('confidence')
axes[0].set_ylabel('frequency')
axes[0].axvline(np.mean(teacher_conf), color='black', linestyle='--', linewidth=2, label=f'mean: {np.mean(teacher_conf):.3f}')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].hist(baseline_conf, bins=20, color='#95a5a6', alpha=0.7, edgecolor='black')
axes[1].set_title('student baseline confidence', fontweight='bold')
axes[1].set_xlabel('confidence')
axes[1].set_ylabel('frequency')
axes[1].axvline(np.mean(baseline_conf), color='black', linestyle='--', linewidth=2, label=f'mean: {np.mean(baseline_conf):.3f}')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

axes[2].hist(distilled_conf, bins=20, color='#27ae60', alpha=0.7, edgecolor='black')
axes[2].set_title('student distilled confidence', fontweight='bold')
axes[2].set_xlabel('confidence')
axes[2].set_ylabel('frequency')
axes[2].axvline(np.mean(distilled_conf), color='black', linestyle='--', linewidth=2, label=f'mean: {np.mean(distilled_conf):.3f}')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("average confidence scores:")
print(f"teacher: {np.mean(teacher_conf):.3f}")
print(f"baseline: {np.mean(baseline_conf):.3f}")
print(f"distilled: {np.mean(distilled_conf):.3f}")

## final results


In [ ]:

print("EXPERIMENT SUMMARY")
print("=" * 60)
print(f"\nmodel sizes:")
print(f"  teacher: {teacher_params:,} params")
print(f"  student: {student_params:,} params")
print(f"  compression: {teacher_params/student_params:.1f}x smaller")
print(f"\nperformance:")
print(f"  teacher: {teacher_acc*100:.2f}%")
print(f"  student baseline: {baseline_acc*100:.2f}%")
print(f"  student distilled: {distilled_acc*100:.2f}%")
print(f"\nkey findings:")
print(f"  - distillation improved student by {(distilled_acc - baseline_acc)*100:.2f} percentage points")
print(f"  - distilled student is {teacher_params/student_params:.1f}x smaller but performs well")
print(f"  - soft targets from teacher help student learn better")
print(f"  - temperature={temperature} and alpha={alpha} worked well for this task")